# RU OCR: PP-OCRv6 small training on Kaggle (2x T4)

One notebook run = one Kaggle session (up to 12 h). Training resumes from the `latest` checkpoint of
the previous session if its output is attached as input.

Run `RUN_NAME` (fine-tuning on all data, 3 epochs, cosine LR without warmup) starts from the **weights** of the
newest checkpoint of the previous run (`INIT_DATASET`, global step 88 000): optimizer, epochs and LR schedule start
anew. Its own checkpoints go to `CKPT_DATASET`; once one of them is attached, it is resumed instead.

**Before the first run**
- Input: the datasets with the packed archives (`synth_part1..5.zip`, `stroyinf.zip`, `stroyinf_modern.zip`,
  `labels.zip`) and `INIT_DATASET` (weights to start from).
- Settings: **Accelerator: GPU T4 x2**, **Internet: on**.
- Add-ons → Secrets: `KAGGLE_API_TOKEN` (your API token) attached to this notebook. With it every new
  `latest` checkpoint is uploaded to the private dataset `CKPT_DATASET` while training runs, so a crash or a
  cancelled run does not lose it (a commit saves `/kaggle/working` only at the end). Without it nothing is uploaded.
- Run with **Save Version → Save & Run All (Commit)**: it runs in the background and keeps `/kaggle/working`.

**Next sessions**
- Add the output of the previous version of this notebook as an input (Add Input → Notebook Output).
  Or attach the `CKPT_DATASET` dataset (always the newest checkpoint, also after a crash).
- Save & Run All again. Of the `latest.pdparams` of this run (`RUN_NAME` folder or `CKPT_DATASET`) found under
  `/kaggle/input` the one with the largest global step is resumed.

`latest` is saved every `SAVE_STEPS` steps (`Global.save_batch_step`, ~15-20 min) and also at the end of
each epoch; resuming from a mid-epoch save continues that epoch, skipping the batches already trained.
Training stops itself after `TIME_LIMIT` so that Kaggle saves the output before its 12 h limit.

In [ ]:
TIME_LIMIT = "680m"     # 11 h 20 min (GNU timeout takes one unit); stop training before the 12 h session limit, outputs are saved after it
BATCH = 32              # first_bs = batch_size_per_card; fp16 at max_w 2400 on a T4 15 GB
WORKERS = 2             # data workers per GPU process (4 CPU cores for 2 processes)
SAVE_STEPS = 2000       # save "latest" every N steps (Global.save_batch_step)
EVAL_STEPS = 10000      # eval on all_val5k every N steps (~90k steps per epoch: 4.13M lines at ~46 lines per step on 2 cards)
SORTED_RATIO = 0.8      # share of batches sorted by w/h: unsorted batches are ~38 % padding; 0.5 in session 1
RUN_NAME = "ru_PP-OCRv6_small_rec_ft3"  # output folder of this run
CKPT_DATASET = "vladislavershov/ru-ocr-checkpoints-ft3"  # private dataset for this run's checkpoint uploads; None: off
INIT_DATASET = "ru-ocr-checkpoints"  # previous run: its newest latest.pdparams gives the initial weights
import os, subprocess
print("CPU cores:", os.cpu_count())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)

## 1. Paddle GPU, code, pretrained weights

In [ ]:
import glob, sys
PADDLE_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"
# Wheels attached as a dataset (pip download ... -d wheels) install offline; otherwise download, the CDN times out
# now and then, so retry.
# Fixed depths, not "**": the data datasets hold millions of crops.
wheels = sorted(w for d in range(1, 6) for w in glob.glob("/kaggle/input" + "/*" * d + "/paddlepaddle_gpu-*.whl"))
if wheels:
    source = ["--no-index", "--find-links", os.path.dirname(wheels[0])]
else:
    source = ["-i", PADDLE_INDEX, "--timeout", "120", "--retries", "10"]
for attempt in range(5):
    pip = [sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", "paddlepaddle-gpu==3.3.1", *source]
    if subprocess.run(pip).returncode == 0:
        break
    print("paddle install attempt", attempt + 1, "failed, retrying", flush=True)
else:
    raise RuntimeError("paddlepaddle-gpu install failed")
import paddle
paddle.utils.run_check()

In [ ]:
%cd /kaggle/working
!rm -rf PaddleOCR && git clone -q --depth 1 --branch feat/ru-ppocrv6-small-training https://github.com/ErshovVE/PaddleOCR.git
%cd /kaggle/working/PaddleOCR
!git log --oneline -1
!pip install -q --progress-bar off -r requirements.txt
!mkdir -p pretrain && wget -q -O pretrain/PP-OCRv6_small_rec_pretrained.pdparams https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_small_rec_pretrained.pdparams

## 2. Data root
Kaggle unpacks the uploaded zips; the walk below finds `<...>/synth/partN` and `<...>/stroyinf/<folder>`
(the folders that hold `crops/`) without entering `crops/`, and links them into `/tmp/ru_root` --
the same layout as the label paths. Archives that were not unpacked are unpacked into `/tmp/ru_root`.

In [ ]:
import zipfile

ROOT = "/tmp/ru_root"
os.makedirs(ROOT, exist_ok=True)
data_dirs, archives, labels_dir, checkpoints = [], [], None, []
for top, dirs, files in os.walk("/kaggle/input"):
    if "crops" in dirs:
        data_dirs.append(top)
        dirs.remove("crops")
    if "synth_2m_train_w_h.txt" in files:
        labels_dir = top
    archives += [os.path.join(top, f) for f in files if f.endswith(".zip")]
    checkpoints += [os.path.join(top, f[: -len(".pdparams")]) for f in files if f == "latest.pdparams"]

for folder in data_dirs:
    group, name = os.path.basename(os.path.dirname(folder)), os.path.basename(folder)
    if group in ("synth", "stroyinf"):
        os.makedirs(os.path.join(ROOT, group), exist_ok=True)
        link = os.path.join(ROOT, group, name)
        if not os.path.exists(link):
            os.symlink(folder, link)
for archive in archives:
    name = os.path.basename(archive)
    if name == "labels.zip" and labels_dir is None:
        zipfile.ZipFile(archive).extractall("/tmp")
        labels_dir = "/tmp/labels"
    elif name != "labels.zip" and not data_dirs:
        print("unpacking", name)
        zipfile.ZipFile(archive).extractall(ROOT)

LABELS = labels_dir
print("data:", sorted(os.listdir(os.path.join(ROOT, "synth"))), len(os.listdir(os.path.join(ROOT, "stroyinf"))), "stroyinf folders")
print("labels:", LABELS, sorted(os.listdir(LABELS)))
missing = 0
for name in ("all_train_w_h.txt", "all_val5k_w_h.txt"):
    with open(os.path.join(LABELS, name)) as f:
        for i, line in enumerate(f):
            if i % 50000 == 0 and not os.path.exists(os.path.join(ROOT, line.split("\t")[0])):
                missing += 1
print("missing crops in a sample of the label lines:", missing)

## 3. Resume or start

In [ ]:
import pickle


def ckpt_step(prefix):
    # global_step from <prefix>.states (file times in /kaggle/input are not reliable)
    try:
        with open(prefix + ".states", "rb") as f:
            return pickle.load(f, encoding="latin1").get("global_step", 0)
    except Exception:
        return -1


def in_folder(prefix, name):
    return name in prefix.split(os.sep)


own = [c for c in checkpoints if in_folder(c, RUN_NAME) or (CKPT_DATASET and in_folder(c, CKPT_DATASET.split("/")[1]))]
init = [c for c in checkpoints if c not in own and in_folder(c, INIT_DATASET)]
for c in checkpoints:
    print("found checkpoint", c, "global step", ckpt_step(c), "(this run)" if c in own else "")
RESUME = max(own, key=ckpt_step) if own else None
INIT = None if RESUME else max(init, key=ckpt_step) if init else None
assert RESUME or INIT, f"attach {CKPT_DATASET} (resume) or {INIT_DATASET} (initial weights)"
print("resume from:", RESUME) if RESUME else print("start from the weights of:", INIT, "global step", ckpt_step(INIT))

## 4. Checkpoint upload
`ru_ocr/tools/ckpt_uploader.py`: a background thread copies `latest.*`, `best_accuracy.*`, `train.log` and `config.yml`
once they stop changing and uploads them as a new version of `CKPT_DATASET` (old versions are deleted).

In [ ]:
sys.path.insert(0, "ru_ocr/tools")
from ckpt_uploader import CheckpointUploader

uploader = None
if CKPT_DATASET:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", "--upgrade", "kaggle"])
        uploader = CheckpointUploader(f"/kaggle/working/output/{RUN_NAME}", "/tmp/ckpt_upload", CKPT_DATASET)
        print("checkpoints are uploaded to", CKPT_DATASET)
    except Exception as exc:
        print("checkpoints are NOT uploaded (no KAGGLE_API_TOKEN secret?):", exc)

## 5. Train
The log goes to `/kaggle/working/train_<time>.log` and to the cell output (every 100 steps).

In [ ]:
import datetime, sys

out_dir = f"/kaggle/working/output/{RUN_NAME}"
log_file = "/kaggle/working/train_{:%Y%m%d_%H%M}.log".format(datetime.datetime.now())
opts = [
    f"Global.save_model_dir={out_dir}",
    f"Global.save_batch_step={SAVE_STEPS}", "Global.save_epoch_step=1",
    "Global.save_on_signal=true",  # TIME_LIMIT stop: finish the step, save latest, exit
    "Global.print_batch_step=100", f"Global.eval_batch_step=[0,{EVAL_STEPS}]",
    "Global.use_amp=true", "Global.amp_dtype=float16", "Global.scale_loss=1024", "Global.use_dynamic_loss_scaling=true",
    f"Train.dataset.data_dir={ROOT}",
    f"Train.dataset.label_file_list=[{LABELS}/all_train_w_h.txt]",
    f"Train.sampler.first_bs={BATCH}", f"Train.sampler.sorted_batch_ratio={SORTED_RATIO}", f"Train.loader.batch_size_per_card={BATCH}", f"Train.loader.num_workers={WORKERS}",
    f"Eval.dataset.data_dir={ROOT}", f"Eval.dataset.label_file_list=[{LABELS}/all_val5k_w_h.txt]",
    "Eval.loader.num_workers=2",
]
if RESUME:
    opts.append(f"Global.checkpoints={RESUME}")
else:
    opts.append(f"Global.pretrained_model={INIT}")  # weights only: new optimizer, epochs and LR schedule
cmd = ["timeout", "--signal=INT", "--kill-after=5m", TIME_LIMIT,
       sys.executable, "-m", "paddle.distributed.launch", "--gpus", "0,1", "--log_dir", "/kaggle/working/dist_log",
       "tools/train.py", "-c", "ru_ocr/configs/ru_PP-OCRv6_small_rec.yml", "-o", *opts]
import time

KEEP = ("global_step", "best metric", "Error", "error", "resume", "RecConAug:", "Traceback", "out of memory")
print(" ".join(cmd))
worker_log = "/kaggle/working/dist_log/workerlog.0"  # rank 0 training log written by paddle.distributed.launch
if uploader is not None:
    uploader.start()
try:
    with open(log_file, "w") as launcher_log:
        proc = subprocess.Popen(cmd, stdout=launcher_log, stderr=subprocess.STDOUT, text=True)
        pos = 0
        while True:
            done = proc.poll() is not None
            if os.path.exists(worker_log):
                with open(worker_log, errors="replace") as f:
                    f.seek(pos)
                    chunk = f.read()
                    pos = f.tell()
                for line in chunk.splitlines():
                    if any(k in line for k in KEEP):
                        print(line, flush=True)
            if done:
                break
            time.sleep(30)
finally:
    if uploader is not None:
        uploader.stop()  # uploads the last checkpoint
        print("checkpoint uploads:", uploader.uploads)
print("exit code", proc.returncode, "(124 = stopped by TIME_LIMIT)")
!tail -20 {log_file}

## 6. What is saved
`latest.*` is the resume point (last save every `SAVE_STEPS` steps or end of epoch), `best_accuracy.*`
the best eval, `iter_epoch_*` one per epoch. The worker logs of `paddle.distributed.launch` are in `dist_log/`.

In [ ]:
!ls -la /kaggle/working/output/ru_PP-OCRv6_small_rec | tail -20
!grep -h "best metric" /kaggle/working/dist_log/workerlog.0 | tail -3